# CTE-Net — sensibilidad a filtrado y rechazo robusto de artefactos

Este notebook responde al Comentario 1. Aplica notch de 50 Hz y filtro Butterworth de fase cero entre 1 y 45 Hz sobre cada registro continuo. Después excluye ventanas con indicadores robustos de amplitud extrema, canales planos, curtosis, actividad frontal lenta/rápida o ruido de línea. Los umbrales se estiman únicamente con participantes de entrenamiento de cada fold mediante una muestra balanceada de 30 ventanas por participante.

El protocolo mantiene los cinco pliegues por participante, los hiperparámetros arquitectónicos corregidos, las diez semillas y la selección del checkpoint mediante validación. **No se ejecuta Optuna.** Además de las métricas, el notebook compara las predicciones y las 342 conexiones dirigidas con los checkpoints originales sobre las mismas ventanas de prueba.

> Antes de ejecutar, añade como fuentes de datos de Kaggle el conjunto EEG y el conjunto que contiene los 50 checkpoints originales. Activa una GPU. Los resultados se guardan después de cada fold y el notebook puede reutilizar folds ya completados.


In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import json
import pickle
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
from scipy import signal, stats
from scipy.stats import binomtest

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================================
# 1. RUTAS
# ============================================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/daprosero/mi-tdah-dataset/"
    "MI_TDAH_Dataset/TDAH"
)
FOLDS_PATH = DATA_ROOT / "folds.pkl"
ADHD_DIR = DATA_ROOT / "ieee/ADHD_group"
CONTROL_DIR = DATA_ROOT / "ieee/Control_group"

# Checkpoints originales: se usan solamente para producir la comparación
# pareada con el modelo publicado. No se modifican ni se reentrenan.
ORIGINAL_MODEL_ROOT = Path(
    "/kaggle/input/datasets/alejandragomezr/models-cte-net/"
    "resultados_hybridtransformer_tekte_tdah_ARTICULO-20260715T225138Z-1-001/"
    "resultados_hybridtransformer_tekte_tdah_ARTICULO"
)

OUTPUT_ROOT = Path('/kaggle/working/cte_net_reviewer2_filter_clean')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================================
# 2. CONDICIÓN DE SENSIBILIDAD
# ============================================================================

EXPERIMENT_NAME = 'filtered_clean_A1A2'
PREPROCESSING_MODE = 'FILTER_CLEAN'

# Frecuencia de muestreo de los archivos originales.
SFREQ = 500.0
WINDOW_SIZE = 512
OVERLAP = 0.50
N_CHANNELS = 19
N_FOLDS = 5
SEEDS = list(range(10))

# Orden de canales utilizado por el conjunto de datos.
CHANNEL_NAMES = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz",
    "C4", "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2",
]


# ============================================================================
# 3. HIPERPARÁMETROS FIJOS DEL MODELO CORREGIDO
# ============================================================================

# Estos son los valores inferidos de los checkpoints originales y reportados
# en la tabla arquitectónica corregida. Este notebook NO ejecuta Optuna.
MODEL_PARAMS = {
    "chans": 19,
    "samples": 512,
    "d_model": 128,
    "nhead": 1,
    "num_transformer_layers": 1,
    "dim_feedforward": 128,
    "transformer_dropout": 0.5,
    "phi_kernel_size": 83,
    "dx": 6,
    "dy": 1,
    "tau": 2,
    "mu": 2,
    "kernel_type": "rational_quadratic",
    "kernel_amplitude": 1.0,
    "kernel_length_scale": 1.0,
    "kernel_alpha": 1.0,
    "clf_hidden": 64,
    "clf_dropout": 0.5,
    "use_pre_transformer_norm": True,
    "use_post_transformer_norm": True,
}

# Configuración de entrenamiento fija. No forma parte de una búsqueda.
# Si el script original empleó otros valores de optimización, cambia solamente
# estas constantes antes de ejecutar; la arquitectura ya está fijada arriba.
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
NUM_WORKERS = 0
USE_AMP = False
RESUME_COMPLETED_FOLDS = True

# Evaluación y análisis de sensibilidad.
PROBABILITY_THRESHOLD = 0.5
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260831
TOP_CONNECTION_FRACTION = 0.10

# Parámetros específicos de la condición filtrada.
FILTER_LOW_HZ = 1.0
FILTER_HIGH_HZ = 45.0
NOTCH_HZ = 50.0
NOTCH_Q = 30.0
FILTER_ORDER = 4
QUALITY_Z_THRESHOLD = 5.0
QUALITY_REFERENCE_WINDOWS_PER_SUBJECT = 30
QUALITY_REFERENCE_SEED = 20260831

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

print("Experimento:", EXPERIMENT_NAME)
print("Modo:", PREPROCESSING_MODE)
print("Dispositivo:", DEVICE)
print("Optuna: DESACTIVADO; se usan hiperparámetros fijos.")


Experimento: filtered_clean_A1A2
Modo: FILTER_CLEAN
Dispositivo: cuda
Optuna: DESACTIVADO; se usan hiperparámetros fijos.


In [2]:
# ============================================================================
# 4. REPRODUCIBILIDAD, FOLDS Y CARGA DE DATOS
# ============================================================================

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def clean_subject_id(value):
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    value = Path(str(value)).name
    if value.lower().endswith(".mat"):
        value = value[:-4]
    return value.strip()


def normalize_folds(raw_folds):
    folds = []
    for fold_index, item in enumerate(raw_folds):
        if isinstance(item, (list, tuple)) and len(item) == 3:
            train_subjects, val_subjects, test_subjects = item
        elif isinstance(item, dict):
            train_subjects = (
                item.get("train") or item.get("train_subjects")
                or item.get("subjects_train")
            )
            val_subjects = (
                item.get("val") or item.get("validation")
                or item.get("val_subjects") or item.get("subjects_val")
            )
            test_subjects = (
                item.get("test") or item.get("test_subjects")
                or item.get("subjects_test")
            )
            if train_subjects is None or val_subjects is None or test_subjects is None:
                raise ValueError(f"No fue posible interpretar el fold {fold_index}.")
        else:
            raise TypeError(f"Formato no reconocido para el fold {fold_index}: {type(item)}")
        folds.append(tuple(
            [clean_subject_id(x) for x in split]
            for split in (train_subjects, val_subjects, test_subjects)
        ))
    return folds


def validate_folds(folds):
    if len(folds) != N_FOLDS:
        raise ValueError(f"Se esperaban {N_FOLDS} folds y se encontraron {len(folds)}.")
    all_subjects = set()
    test_counter = Counter()
    for fold_index, (train_subjects, val_subjects, test_subjects) in enumerate(folds):
        train_set, val_set, test_set = map(set, (train_subjects, val_subjects, test_subjects))
        if train_set & val_set or train_set & test_set or val_set & test_set:
            raise ValueError(f"El fold {fold_index} contiene participantes en más de una partición.")
        all_subjects.update(train_set | val_set | test_set)
        test_counter.update(test_subjects)
    if len(all_subjects) != 120:
        raise ValueError(f"Se esperaban 120 participantes y se encontraron {len(all_subjects)}.")
    invalid = {subject: count for subject, count in test_counter.items() if count != 1}
    if invalid:
        raise ValueError(f"Cada participante debe aparecer una vez en test: {invalid}")
    return all_subjects


def build_mat_index(directory):
    directory = Path(directory)
    if not directory.exists():
        raise FileNotFoundError(f"No existe la carpeta de datos: {directory}")
    files = {path.stem: path for path in sorted(directory.glob("*.mat"))}
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos .mat en: {directory}")
    return files


def load_subject_eeg(mat_path, subject_id):
    mat = scipy.io.loadmat(mat_path)
    if subject_id in mat:
        variable_name = subject_id
        raw = np.asarray(mat[subject_id])
    else:
        candidates = []
        for key, value in mat.items():
            if key.startswith("__"):
                continue
            array = np.asarray(value)
            if array.ndim == 2 and np.issubdtype(array.dtype, np.number) and N_CHANNELS in array.shape:
                candidates.append((key, array))
        if not candidates:
            available = {k: np.asarray(v).shape for k, v in mat.items() if not k.startswith("__")}
            raise ValueError(f"No se encontró una matriz EEG válida en {mat_path}: {available}")
        variable_name, raw = max(candidates, key=lambda item: item[1].size)

    raw = np.asarray(raw, dtype=np.float32)
    eeg = raw.T
    if eeg.shape[0] != N_CHANNELS and raw.shape[0] == N_CHANNELS:
        eeg = raw
    if eeg.shape[0] != N_CHANNELS:
        raise ValueError(f"{subject_id}: se esperaban {N_CHANNELS} canales y se obtuvo {eeg.shape}.")
    if not np.isfinite(eeg).all():
        raise ValueError(f"{subject_id}: la señal contiene NaN o Inf.")
    return eeg.astype(np.float32), variable_name


def filter_continuous_eeg(eeg):
    """Notch de 50 Hz seguido de Butterworth 1--45 Hz, ambos con fase cero."""
    b_notch, a_notch = signal.iirnotch(NOTCH_HZ, NOTCH_Q, fs=SFREQ)
    notched = signal.filtfilt(b_notch, a_notch, eeg, axis=-1)
    sos = signal.butter(
        FILTER_ORDER,
        [FILTER_LOW_HZ, FILTER_HIGH_HZ],
        btype="bandpass",
        fs=SFREQ,
        output="sos",
    )
    filtered = signal.sosfiltfilt(sos, notched, axis=-1)
    return np.asarray(filtered, dtype=np.float32)


def common_average_reference(eeg):
    """Referencia promedio común calculada muestra a muestra entre 19 electrodos."""
    return np.asarray(eeg - eeg.mean(axis=0, keepdims=True), dtype=np.float32)


def transform_recording(eeg):
    if PREPROCESSING_MODE == "FILTER_CLEAN":
        return filter_continuous_eeg(eeg)
    if PREPROCESSING_MODE == "CAR":
        return common_average_reference(eeg)
    raise ValueError(f"PREPROCESSING_MODE no reconocido: {PREPROCESSING_MODE}")


def load_and_segment_all_subjects():
    with open(FOLDS_PATH, "rb") as file:
        folds = normalize_folds(pickle.load(file))
    protocol_subjects = validate_folds(folds)

    adhd_files = build_mat_index(ADHD_DIR)
    control_files = build_mat_index(CONTROL_DIR)
    if set(adhd_files) & set(control_files):
        raise ValueError("Hay participantes presentes en ambas clases.")
    missing = protocol_subjects - (set(adhd_files) | set(control_files))
    if missing:
        raise FileNotFoundError(f"Faltan archivos para: {sorted(missing)}")

    control_subjects = sorted(protocol_subjects & set(control_files))
    adhd_subjects = sorted(protocol_subjects & set(adhd_files))
    if len(control_subjects) != 60 or len(adhd_subjects) != 60:
        raise ValueError("La cohorte debe contener 60 controles y 60 participantes ADHD.")

    records = (
        [(subject, 0, "Control", control_files[subject]) for subject in control_subjects]
        + [(subject, 1, "ADHD", adhd_files[subject]) for subject in adhd_subjects]
    )
    stride = int(round(WINDOW_SIZE * (1.0 - OVERLAP)))

    raw_windows, condition_windows, labels, subject_ids, window_ids = [], [], [], [], []
    metadata_rows = []
    global_window_index = 0

    for subject_position, (subject_id, label, class_name, mat_path) in enumerate(records):
        raw_eeg, variable_name = load_subject_eeg(mat_path, subject_id)
        condition_eeg = transform_recording(raw_eeg)
        n_samples = raw_eeg.shape[1]
        if n_samples < WINDOW_SIZE:
            raise ValueError(f"{subject_id}: solo tiene {n_samples} muestras.")

        local_window = 0
        for start in range(0, n_samples - WINDOW_SIZE + 1, stride):
            end = start + WINDOW_SIZE
            raw_windows.append(raw_eeg[:, start:end])
            condition_windows.append(condition_eeg[:, start:end])
            labels.append(label)
            subject_ids.append(subject_id)
            window_ids.append(local_window)
            metadata_rows.append({
                "global_window_index": global_window_index,
                "subject_position": subject_position,
                "subject_id": subject_id,
                "label": label,
                "class_name": class_name,
                "window_id": local_window,
                "start_sample": start,
                "end_sample": end,
                "mat_variable": variable_name,
                "mat_filename": mat_path.name,
            })
            global_window_index += 1
            local_window += 1

    data = {
        "X_raw": np.stack(raw_windows).astype(np.float32),
        "X_condition": np.stack(condition_windows).astype(np.float32),
        "y": np.asarray(labels, dtype=np.int64),
        "subject_ids": np.asarray(subject_ids, dtype=str),
        "window_ids": np.asarray(window_ids, dtype=np.int64),
        "metadata": pd.DataFrame(metadata_rows),
        "folds": folds,
    }
    print("X raw:", data["X_raw"].shape)
    print("X condición:", data["X_condition"].shape)
    print("Participantes:", np.unique(data["subject_ids"]).size)
    return data


# ============================================================================
# 5. CONTROL ROBUSTO DE CALIDAD PARA LA CONDICIÓN FILTRADA
# ============================================================================

QUALITY_FEATURE_NAMES = [
    "max_abs_amplitude",
    "max_peak_to_peak",
    "max_abs_kurtosis",
    "frontal_low_frequency_ratio",
    "frontal_high_frequency_ratio",
    "line_noise_ratio",
    "min_channel_std",
]


def _band_power(psd, frequencies, low, high):
    mask = (frequencies >= low) & (frequencies <= high)
    if not mask.any():
        return np.zeros(psd.shape[:-1], dtype=np.float64)
    return psd[..., mask].sum(axis=-1)


def compute_quality_features(X, batch_size=256):
    """Indicadores independientes de la etiqueta calculados sobre EEG original."""
    frames = []
    frontal_indices = [CHANNEL_NAMES.index("Fp1"), CHANNEL_NAMES.index("Fp2")]
    for start in range(0, len(X), batch_size):
        batch = np.asarray(X[start:start + batch_size], dtype=np.float64)
        channel_std = batch.std(axis=-1)
        peak_to_peak = np.ptp(batch, axis=-1)
        kurt = stats.kurtosis(batch, axis=-1, fisher=True, bias=False, nan_policy="omit")
        frequencies, psd = signal.welch(
            batch,
            fs=SFREQ,
            nperseg=min(256, WINDOW_SIZE),
            axis=-1,
            detrend="constant",
        )
        total_power = _band_power(psd, frequencies, 1.0, 45.0) + 1e-12
        low_ratio = _band_power(psd, frequencies, 1.0, 4.0) / total_power
        high_ratio = _band_power(psd, frequencies, 30.0, 45.0) / total_power
        line_denom = _band_power(psd, frequencies, 45.0, 55.0) + 1e-12
        line_ratio = _band_power(psd, frequencies, 49.0, 51.0) / line_denom

        frames.append(pd.DataFrame({
            "max_abs_amplitude": np.max(np.abs(batch), axis=(1, 2)),
            "max_peak_to_peak": np.max(peak_to_peak, axis=1),
            "max_abs_kurtosis": np.nanmax(np.abs(kurt), axis=1),
            "frontal_low_frequency_ratio": np.nanmax(low_ratio[:, frontal_indices], axis=1),
            "frontal_high_frequency_ratio": np.nanmax(high_ratio[:, frontal_indices], axis=1),
            "line_noise_ratio": np.nanmax(line_ratio, axis=1),
            "min_channel_std": np.nanmin(channel_std, axis=1),
            "nonfinite": ~np.isfinite(batch).all(axis=(1, 2)),
            "flat_channel": np.nanmin(peak_to_peak, axis=1) <= 1e-12,
        }))
    return pd.concat(frames, ignore_index=True)


def robust_center_scale(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale <= 1e-12:
        q25, q75 = np.percentile(values, [25, 75])
        scale = float((q75 - q25) / 1.349)
    if not np.isfinite(scale) or scale <= 1e-12:
        scale = float(np.std(values))
    if not np.isfinite(scale) or scale <= 1e-12:
        scale = 1e-12
    return median, scale


def participant_balanced_reference_indices(subject_ids, train_subjects, fold_index):
    rng = np.random.default_rng(QUALITY_REFERENCE_SEED + fold_index)
    selected = []
    for subject in sorted(train_subjects):
        candidates = np.flatnonzero(subject_ids == subject)
        n_take = min(QUALITY_REFERENCE_WINDOWS_PER_SUBJECT, len(candidates))
        selected.extend(rng.choice(candidates, size=n_take, replace=False).tolist())
    return np.asarray(selected, dtype=np.int64)


def build_cleaning_mask(quality_features, subject_ids, train_subjects, fold_index):
    reference_indices = participant_balanced_reference_indices(
        subject_ids, train_subjects, fold_index
    )
    reference = quality_features.iloc[reference_indices]
    flags = quality_features["nonfinite"].to_numpy(bool) | quality_features["flat_channel"].to_numpy(bool)
    threshold_rows = []

    for feature in QUALITY_FEATURE_NAMES:
        median, scale = robust_center_scale(reference[feature].to_numpy())
        values = quality_features[feature].to_numpy(dtype=np.float64)
        if feature == "min_channel_std":
            robust_z = (median - values) / scale
            direction = "lower"
        else:
            robust_z = (values - median) / scale
            direction = "upper"
        current = (~np.isfinite(values)) | (robust_z > QUALITY_Z_THRESHOLD)
        flags |= current
        threshold_rows.append({
            "fold": fold_index,
            "feature": feature,
            "reference_median": median,
            "reference_robust_scale": scale,
            "z_threshold": QUALITY_Z_THRESHOLD,
            "direction": direction,
            "n_reference_windows": len(reference_indices),
            "n_flagged_all_windows": int(current.sum()),
        })
    return ~flags, pd.DataFrame(threshold_rows)


def prepare_fold_partitions(data):
    quality_features = None
    if PREPROCESSING_MODE == "FILTER_CLEAN":
        print("Calculando indicadores de calidad sobre las ventanas originales...")
        quality_features = compute_quality_features(data["X_raw"])
        quality_features.insert(0, "global_window_index", np.arange(len(quality_features)))
        quality_features.to_csv(OUTPUT_ROOT / "quality_features_all_windows.csv", index=False)

    partitions = {}
    quality_rows, threshold_frames = [], []
    for fold_index, (train_subjects, val_subjects, test_subjects) in enumerate(data["folds"]):
        if PREPROCESSING_MODE == "FILTER_CLEAN":
            keep, thresholds = build_cleaning_mask(
                quality_features, data["subject_ids"], train_subjects, fold_index
            )
            threshold_frames.append(thresholds)
        else:
            keep = np.ones(len(data["y"]), dtype=bool)

        split_indices = {}
        for split_name, split_subjects in (
            ("train", train_subjects), ("validation", val_subjects), ("test", test_subjects)
        ):
            base = np.isin(data["subject_ids"], np.asarray(split_subjects, dtype=str))
            indices = np.flatnonzero(base & keep)
            split_indices[split_name] = indices
            quality_rows.append({
                "fold": fold_index,
                "split": split_name,
                "n_subjects": len(np.unique(data["subject_ids"][base])),
                "n_windows_before": int(base.sum()),
                "n_windows_retained": int((base & keep).sum()),
                "n_windows_flagged": int((base & ~keep).sum()),
                "percent_flagged": 100.0 * float((base & ~keep).sum()) / max(1, int(base.sum())),
            })
            retained_by_subject = pd.Series(data["subject_ids"][indices]).value_counts()
            missing_subjects = set(split_subjects) - set(retained_by_subject.index)
            if missing_subjects:
                raise RuntimeError(
                    f"Fold {fold_index}, {split_name}: participantes sin ventanas retenidas: {sorted(missing_subjects)}"
                )
        partitions[fold_index] = split_indices

    pd.DataFrame(quality_rows).to_csv(OUTPUT_ROOT / "quality_summary_by_fold_and_split.csv", index=False)
    if threshold_frames:
        pd.concat(threshold_frames, ignore_index=True).to_csv(
            OUTPUT_ROOT / "quality_thresholds_by_fold.csv", index=False
        )
    return partitions


In [3]:
# ============================================================================
# 6. ARQUITECTURA CTE-NET
# ============================================================================

class InspectableTransformerEncoder(nn.Module):
    def __init__(self, in_channels, d_model, num_heads, intermediate_dim, dropout):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model debe ser divisible por num_heads.")
        self.input_proj = nn.Linear(in_channels, d_model)
        self.self_attention = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.self_attention_layer_norm = nn.LayerNorm(d_model, eps=1e-6)
        self.self_attention_dropout = nn.Dropout(dropout)
        self.feedforward_intermediate_dense = nn.Linear(d_model, intermediate_dim)
        self.feedforward_output_dense = nn.Linear(intermediate_dim, d_model)
        self.feedforward_dropout = nn.Dropout(dropout)
        self.feedforward_layer_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, inputs):
        x = self.input_proj(inputs)
        attention_output, _ = self.self_attention(x, x, x, need_weights=False)
        attention_output = self.self_attention_dropout(attention_output)
        attention_output = self.self_attention_layer_norm(x + attention_output)
        ff = self.feedforward_intermediate_dense(attention_output)
        ff = F.gelu(ff)
        ff = self.feedforward_output_dense(ff)
        ff = self.feedforward_dropout(ff)
        return self.feedforward_layer_norm(attention_output + ff)


class ChannelwiseTemporalFilter(nn.Module):
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, stride=1, padding=0, groups=channels),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=4, stride=4),
            nn.BatchNorm1d(channels),
        )

    def forward(self, x):
        return self.net(x)


class TakensConv1D(nn.Module):
    def __init__(self, dx, dy, tau, mu):
        super().__init__()
        self.dx, self.dy, self.tau, self.mu = map(int, (dx, dy, tau, mu))
        self.num_filters = self.dx + self.dy + 1
        kernel_size = self.mu + (self.dx - 1) * self.tau + 1
        kernel = torch.zeros(self.num_filters, 1, kernel_size)
        for i in range(self.dx):
            kernel[i, 0, self.mu + i * self.tau] = 1.0
        for i in range(self.dy):
            kernel[self.dx + i, 0, (i + 1) * self.tau] = 1.0
        kernel[self.dx + self.dy, 0, 0] = 1.0
        self.register_buffer("kernel", torch.flip(kernel, dims=[-1]))

    def forward(self, inputs):
        batch_size, channels, time_length = inputs.shape
        reshaped = inputs.reshape(batch_size * channels, 1, time_length)
        conv = F.conv1d(reshaped, self.kernel, stride=self.tau)
        length = conv.shape[-1]
        output = conv.reshape(batch_size, channels, self.num_filters, length).permute(0, 1, 3, 2)
        return (
            output[..., :self.dx],
            output[..., self.dx:self.dx + self.dy],
            output[..., -1:],
        )


class KernelLayer(nn.Module):
    def __init__(self, amplitude=1.0, length_scale=1.0, alpha=1.0, kernel_type="rational_quadratic"):
        super().__init__()
        self.kernel_type = kernel_type.lower()
        self.register_buffer("amplitude", torch.tensor(float(amplitude)))
        self.register_buffer("length_scale", torch.tensor(float(length_scale)))
        if self.kernel_type == "rational_quadratic":
            self.register_buffer("alpha", torch.tensor(float(alpha)))

    def forward(self, X):
        distance_squared = ((X.unsqueeze(-2) - X.unsqueeze(-3)) ** 2).sum(dim=-1)
        length_scale = torch.clamp(self.length_scale, min=1e-8)
        if self.kernel_type == "gaussian":
            return self.amplitude ** 2 * torch.exp(-distance_squared / (2.0 * length_scale ** 2))
        alpha = torch.clamp(self.alpha, min=1e-8)
        return self.amplitude ** 2 * (
            1.0 + distance_squared / (2.0 * alpha * length_scale ** 2)
        ) ** (-alpha)


class TransferEntropyLayer(nn.Module):
    def __init__(self, alpha=2):
        super().__init__()
        self.alpha = int(alpha)

    def compute_entropy(self, K):
        trace = torch.diagonal(K, dim1=-2, dim2=-1).sum(dim=-1).unsqueeze(-1).unsqueeze(-1) + 1e-8
        normalized = K / trace
        if self.alpha == 2:
            power = normalized @ normalized
            trace_power = torch.diagonal(power, dim1=-2, dim2=-1).sum(dim=-1)
            return -torch.log(trace_power + 1e-8)
        eigenvalues = torch.clamp(torch.linalg.eigvalsh(normalized).real, min=1e-8)
        return torch.log((eigenvalues ** self.alpha).sum(dim=-1) + 1e-8) / (1 - self.alpha)

    def forward(self, K_x, K_y_minus_1, K_y):
        K_x = K_x.unsqueeze(2)
        K_y_minus_1 = K_y_minus_1.unsqueeze(1)
        K_y = K_y.unsqueeze(1)
        return (
            self.compute_entropy(K_y_minus_1 * K_x)
            - self.compute_entropy(K_y * K_y_minus_1 * K_x)
            + self.compute_entropy(K_y * K_y_minus_1)
            - self.compute_entropy(K_y_minus_1)
        )


class RemoveDiagonalFlatten(nn.Module):
    def forward(self, inputs):
        batch_size, channels, channels_2 = inputs.shape
        if channels != channels_2:
            raise ValueError("La matriz TE debe ser cuadrada.")
        mask = ~torch.eye(channels, dtype=torch.bool, device=inputs.device)
        return inputs[:, mask].reshape(batch_size, channels * (channels - 1))


class HybridTransformerTEKTE(nn.Module):
    def __init__(self, **params):
        super().__init__()
        self.chans = params["chans"]
        self.samples = params["samples"]
        self.pre_transformer_norm = (
            nn.LayerNorm(self.chans) if params["use_pre_transformer_norm"] else nn.Identity()
        )
        layers = []
        for layer_index in range(params["num_transformer_layers"]):
            layers.append(InspectableTransformerEncoder(
                in_channels=self.chans if layer_index == 0 else params["d_model"],
                d_model=params["d_model"],
                num_heads=params["nhead"],
                intermediate_dim=params["dim_feedforward"],
                dropout=params["transformer_dropout"],
            ))
        self.transformer_layers = nn.ModuleList(layers)
        self.post_transformer_norm = (
            nn.LayerNorm(params["d_model"]) if params["use_post_transformer_norm"] else nn.Identity()
        )
        self.proj_back = nn.Linear(params["d_model"], self.chans)
        self.phi = ChannelwiseTemporalFilter(self.chans, params["phi_kernel_size"])
        self.takens = TakensConv1D(params["dx"], params["dy"], params["tau"], params["mu"])
        self.dense_proj_x = nn.Linear(params["dx"], params["dx"], bias=False)
        self.dense_proj_y1 = nn.Linear(params["dy"], params["dy"], bias=False)
        self.dense_proj_y = nn.Linear(1, 1, bias=False)
        kernel_args = dict(
            amplitude=params["kernel_amplitude"],
            length_scale=params["kernel_length_scale"],
            alpha=params["kernel_alpha"],
            kernel_type=params["kernel_type"],
        )
        self.kernel_x = KernelLayer(**kernel_args)
        self.kernel_y_minus_1 = KernelLayer(**kernel_args)
        self.kernel_y = KernelLayer(**kernel_args)
        self.transfer_entropy = TransferEntropyLayer(alpha=2)
        self.remove_diag_flatten = RemoveDiagonalFlatten()
        self.classifier = nn.Sequential(
            nn.Linear(self.chans * (self.chans - 1), params["clf_hidden"]),
            nn.ReLU(),
            nn.Dropout(params["clf_dropout"]),
            nn.Linear(params["clf_hidden"], 1),
        )

    def forward(self, x, return_dict=True):
        if x.shape[1:] != (self.chans, self.samples):
            raise ValueError(f"Entrada esperada (*, {self.chans}, {self.samples}); recibida {tuple(x.shape)}")
        hidden = self.pre_transformer_norm(x.transpose(1, 2))
        for layer in self.transformer_layers:
            hidden = layer(hidden)
        hidden = self.post_transformer_norm(hidden)
        Xf = self.proj_back(hidden).transpose(1, 2)
        phi = self.phi(Xf)
        x_sub, y_minus_1, y_t = self.takens(phi)
        K_x = self.kernel_x(self.dense_proj_x(x_sub))
        K_y_minus_1 = self.kernel_y_minus_1(self.dense_proj_y1(y_minus_1))
        K_y = self.kernel_y(self.dense_proj_y(y_t))
        transfer_entropy = self.transfer_entropy(K_x, K_y_minus_1, K_y)
        features = self.remove_diag_flatten(transfer_entropy)
        logits = self.classifier(features).squeeze(-1)
        if return_dict:
            return {
                "logits": logits,
                "probs": torch.sigmoid(logits),
                "T": transfer_entropy,
                "features": features,
                "Xf": Xf,
                "phi": phi,
            }
        return logits


def build_model():
    return HybridTransformerTEKTE(**MODEL_PARAMS)


model_check = build_model()
n_parameters = sum(parameter.numel() for parameter in model_check.parameters() if parameter.requires_grad)
with torch.inference_mode():
    check_output = model_check(torch.zeros(1, N_CHANNELS, WINDOW_SIZE), return_dict=True)
print("Parámetros entrenables:", f"{n_parameters:,}")
print("T_phi:", check_output["phi"].shape[-1])
takens_kernel_size = MODEL_PARAMS["mu"] + (MODEL_PARAMS["dx"] - 1) * MODEL_PARAMS["tau"] + 1
t_z = (check_output["phi"].shape[-1] - takens_kernel_size) // MODEL_PARAMS["tau"] + 1
print("T_z:", t_z)
print("Coeficientes dirigidos fuera de la diagonal:", check_output["features"].shape[-1])
assert n_parameters == 128_578, f"Número de parámetros inesperado: {n_parameters}"
assert check_output["phi"].shape[-1] == 107
assert t_z == 48
assert check_output["T"].shape[-2:] == (19, 19)
del model_check, check_output


Parámetros entrenables: 128,578
T_phi: 107
T_z: 48
Coeficientes dirigidos fuera de la diagonal: 342


In [4]:
# ============================================================================
# 7. CHECKPOINTS ORIGINALES, ENTRENAMIENTO E INFERENCIA
# ============================================================================

def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)


def load_state_dict_file(path):
    loaded = safe_torch_load(path)
    if not isinstance(loaded, dict):
        raise TypeError(f"Checkpoint no reconocido: {path}")
    if "model_state_dict" in loaded:
        state_dict = loaded["model_state_dict"]
    elif "state_dict" in loaded:
        state_dict = loaded["state_dict"]
    else:
        state_dict = loaded
    cleaned = {}
    for key, value in state_dict.items():
        clean_key = key
        for prefix in ("module.", "_orig_mod."):
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]
        cleaned[clean_key] = value
    return cleaned


def extract_seed_from_path(path):
    for pattern in (
        r"repeat_seed_(\d+)", r"fixedparams[^/\\]*seed_(\d+)",
        r"fixed_seed_(\d+)", r"seed[_-](\d+)",
    ):
        match = re.search(pattern, str(path), flags=re.IGNORECASE)
        if match:
            return int(match.group(1))
    return None


def extract_fold_from_path(path):
    matches = re.findall(r"fold_(\d+)", str(path), flags=re.IGNORECASE)
    return int(matches[-1]) if matches else None


def checkpoint_priority(path):
    priorities = {
        "best_state.pt": 100,
        "final_state_dict.pt": 80,
        "resume_checkpoint.pt": 20,
    }
    if path.name.lower().endswith("state_dict.pt"):
        return max(70, priorities.get(path.name.lower(), 0))
    return priorities.get(path.name.lower(), 0)


def discover_original_checkpoints():
    if not ORIGINAL_MODEL_ROOT.exists():
        raise FileNotFoundError(f"No existe la carpeta de checkpoints: {ORIGINAL_MODEL_ROOT}")
    candidates = []
    for pattern in ("best_state.pt", "final_state_dict.pt", "*state_dict.pt", "resume_checkpoint.pt"):
        candidates.extend(ORIGINAL_MODEL_ROOT.rglob(pattern))
    grouped = {}
    for path in sorted(set(candidates), key=str):
        seed, fold = extract_seed_from_path(path), extract_fold_from_path(path)
        if seed in SEEDS and fold is not None and 0 <= fold < N_FOLDS:
            grouped.setdefault((seed, fold), []).append(path)
    mapping = {key: max(paths, key=checkpoint_priority) for key, paths in grouped.items()}
    expected = {(seed, fold) for seed in SEEDS for fold in range(N_FOLDS)}
    missing = sorted(expected - set(mapping))
    if missing:
        raise FileNotFoundError(f"Faltan checkpoints originales: {missing}")
    print("Checkpoints originales encontrados:", len(mapping))
    return mapping


def make_loader(X, y, indices, shuffle, seed, batch_size=BATCH_SIZE):
    indices = np.asarray(indices, dtype=np.int64)
    dataset = TensorDataset(
        torch.from_numpy(np.asarray(X[indices], dtype=np.float32)),
        torch.from_numpy(np.asarray(y[indices], dtype=np.float32)),
        torch.from_numpy(indices),
    )
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=generator,
    )


def binary_metrics(y_true, y_prob, threshold=PROBABILITY_THRESHOLD):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= threshold).astype(np.int64)
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    safe = lambda a, b: float(a / b) if b else np.nan
    sensitivity = safe(tp, tp + fn)
    specificity = safe(tn, tn + fp)
    precision = safe(tp, tp + fp)
    f1 = safe(2 * precision * sensitivity, precision + sensitivity)
    try:
        auc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        auc = np.nan
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1,
        "roc_auc": auc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def evaluate_loader(model, loader, criterion=None, collect_te=True):
    model.eval()
    probabilities, labels, global_indices, features = [], [], [], []
    total_loss, n_samples = 0.0, 0
    with torch.inference_mode():
        for batch_x, batch_y, batch_indices in loader:
            batch_x = batch_x.to(DEVICE, dtype=torch.float32, non_blocking=True)
            batch_y = batch_y.to(DEVICE, dtype=torch.float32, non_blocking=True)
            output = model(batch_x, return_dict=True)
            if criterion is not None:
                loss = criterion(output["logits"], batch_y)
                total_loss += float(loss.item()) * len(batch_y)
            n_samples += len(batch_y)
            probabilities.append(output["probs"].cpu().numpy())
            labels.append(batch_y.cpu().numpy())
            global_indices.append(batch_indices.numpy())
            if collect_te:
                features.append(output["features"].cpu().numpy().astype(np.float32))
    result = {
        "probabilities": np.concatenate(probabilities),
        "labels": np.concatenate(labels).astype(np.int64),
        "global_indices": np.concatenate(global_indices).astype(np.int64),
        "mean_loss": total_loss / max(1, n_samples) if criterion is not None else np.nan,
    }
    result["features"] = np.concatenate(features) if collect_te else None
    return result


def train_condition_model(data, partitions, seed, fold):
    checkpoint_dir = CHECKPOINT_ROOT / f"seed_{seed}" / f"fold_{fold}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "best_state.pt"
    history_path = checkpoint_dir / "training_history.csv"

    model = build_model().to(DEVICE)
    if RESUME_COMPLETED_FOLDS and checkpoint_path.exists():
        saved = safe_torch_load(checkpoint_path)
        if saved.get("experiment_name") != EXPERIMENT_NAME:
            raise RuntimeError(f"Checkpoint incompatible encontrado en {checkpoint_path}")
        model.load_state_dict(saved["model_state_dict"], strict=True)
        history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
        print(f"seed={seed} fold={fold}: checkpoint reutilizado")
        return model, history, int(saved["best_epoch"]), float(saved["best_val_accuracy"])

    set_seed(seed * 100 + fold)
    model = build_model().to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda")
    train_loader = make_loader(
        data["X_condition"], data["y"], partitions[fold]["train"],
        shuffle=True, seed=seed * 100 + fold,
    )
    val_loader = make_loader(
        data["X_condition"], data["y"], partitions[fold]["validation"],
        shuffle=False, seed=seed * 100 + fold,
    )

    best_accuracy, best_loss, best_epoch, no_improvement = -np.inf, np.inf, -1, 0
    best_state = None
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum, train_count = 0.0, 0
        for batch_x, batch_y, _ in train_loader:
            batch_x = batch_x.to(DEVICE, dtype=torch.float32, non_blocking=True)
            batch_y = batch_y.to(DEVICE, dtype=torch.float32, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP and DEVICE.type == "cuda",
            ):
                output = model(batch_x, return_dict=True)
                loss = criterion(output["logits"], batch_y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss_sum += float(loss.item()) * len(batch_y)
            train_count += len(batch_y)

        validation = evaluate_loader(model, val_loader, criterion=criterion, collect_te=False)
        val_metrics = binary_metrics(validation["labels"], validation["probabilities"])
        train_loss = train_loss_sum / max(1, train_count)
        val_accuracy = val_metrics["accuracy"]
        val_loss = validation["mean_loss"]
        history_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_sensitivity": val_metrics["sensitivity"],
            "val_specificity": val_metrics["specificity"],
        })

        improved = (val_accuracy > best_accuracy + 1e-12) or (
            np.isclose(val_accuracy, best_accuracy) and val_loss < best_loss - 1e-12
        )
        if improved:
            best_accuracy, best_loss, best_epoch = val_accuracy, val_loss, epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            no_improvement = 0
        else:
            no_improvement += 1

        if epoch == 1 or epoch % 5 == 0 or improved:
            print(
                f"seed={seed} fold={fold} epoch={epoch:03d} "
                f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                f"val_acc={val_accuracy:.4f} best={best_accuracy:.4f}"
            )
        if no_improvement >= EARLY_STOPPING_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError("No se produjo un estado válido durante el entrenamiento.")
    history = pd.DataFrame(history_rows)
    history.to_csv(history_path, index=False)
    torch.save({
        "model_state_dict": best_state,
        "experiment_name": EXPERIMENT_NAME,
        "preprocessing_mode": PREPROCESSING_MODE,
        "seed": seed,
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_accuracy,
        "best_val_loss": best_loss,
        "model_params": MODEL_PARAMS,
        "training_config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "max_epochs": MAX_EPOCHS,
            "patience": EARLY_STOPPING_PATIENCE,
        },
    }, checkpoint_path)
    model.load_state_dict(best_state, strict=True)
    return model, history, best_epoch, best_accuracy


def aggregate_test_outputs(result, data, seed, fold, condition):
    indices = result["global_indices"]
    frame = pd.DataFrame({
        "condition": condition,
        "seed": seed,
        "fold": fold,
        "global_window_index": indices,
        "subject_id": data["subject_ids"][indices],
        "window_id": data["window_ids"][indices],
        "label": result["labels"],
        "prob_adhd": result["probabilities"],
    })
    frame["prediction"] = (frame["prob_adhd"] >= PROBABILITY_THRESHOLD).astype(int)
    frame["correct"] = (frame["prediction"] == frame["label"]).astype(int)

    subject_frame = frame.groupby(
        ["condition", "seed", "fold", "subject_id", "label"], as_index=False
    ).agg(
        n_windows=("prob_adhd", "size"),
        mean_prob_adhd=("prob_adhd", "mean"),
        std_prob_adhd=("prob_adhd", "std"),
        window_accuracy=("correct", "mean"),
    )

    te_by_subject = {}
    for subject in subject_frame["subject_id"]:
        local = np.flatnonzero(frame["subject_id"].to_numpy() == subject)
        te_by_subject[str(subject)] = result["features"][local].mean(axis=0).astype(np.float32)
    return frame, subject_frame, te_by_subject


In [5]:
# ============================================================================
# 8. IC DEL 95 %, COMPARACIÓN PAREADA Y ESTABILIDAD DE CONECTIVIDAD
# ============================================================================

METRIC_NAMES = ["accuracy", "sensitivity", "specificity", "precision", "f1_score", "roc_auc"]


def participant_metrics_by_seed(subject_predictions):
    rows = []
    for (condition, seed), frame in subject_predictions.groupby(["condition", "seed"]):
        metrics = binary_metrics(frame["label"], frame["mean_prob_adhd"])
        rows.append({"condition": condition, "seed": seed, **{m: metrics[m] for m in METRIC_NAMES}})
    return pd.DataFrame(rows)


def stratified_participant_bootstrap(subject_predictions, n_bootstrap=N_BOOTSTRAP):
    summaries, distributions = [], []
    for condition, frame in subject_predictions.groupby("condition"):
        probabilities = frame.pivot(index="subject_id", columns="seed", values="mean_prob_adhd").sort_index()
        labels = (
            frame.drop_duplicates("subject_id").set_index("subject_id")["label"]
            .reindex(probabilities.index).to_numpy(dtype=np.int64)
        )
        prob = probabilities.to_numpy(dtype=np.float64)
        control = np.flatnonzero(labels == 0)
        adhd = np.flatnonzero(labels == 1)
        rng = np.random.default_rng(BOOTSTRAP_SEED)

        seed_metrics = []
        for seed_position in range(prob.shape[1]):
            values = binary_metrics(labels, prob[:, seed_position])
            seed_metrics.append([values[m] for m in METRIC_NAMES])
        point = np.nanmean(np.asarray(seed_metrics), axis=0)

        boot = np.empty((n_bootstrap, len(METRIC_NAMES)), dtype=np.float64)
        for replicate in range(n_bootstrap):
            sampled = np.concatenate([
                rng.choice(control, size=len(control), replace=True),
                rng.choice(adhd, size=len(adhd), replace=True),
            ])
            replicate_metrics = []
            for seed_position in range(prob.shape[1]):
                values = binary_metrics(labels[sampled], prob[sampled, seed_position])
                replicate_metrics.append([values[m] for m in METRIC_NAMES])
            boot[replicate] = np.nanmean(np.asarray(replicate_metrics), axis=0)

        lower, upper = np.nanpercentile(boot, [2.5, 97.5], axis=0)
        for metric_index, metric in enumerate(METRIC_NAMES):
            summaries.append({
                "condition": condition,
                "metric": metric,
                "estimate": point[metric_index],
                "ci_95_lower": lower[metric_index],
                "ci_95_upper": upper[metric_index],
                "n_subjects": len(labels),
                "n_seeds": prob.shape[1],
                "n_bootstrap": n_bootstrap,
            })
        distributions.append(pd.DataFrame(boot, columns=METRIC_NAMES).assign(
            condition=condition,
            bootstrap_replicate=np.arange(1, n_bootstrap + 1),
        ))
    return pd.DataFrame(summaries), pd.concat(distributions, ignore_index=True)


def paired_metric_difference_bootstrap(subject_predictions, original_name, condition_name):
    original = subject_predictions[subject_predictions["condition"] == original_name]
    condition = subject_predictions[subject_predictions["condition"] == condition_name]
    original_prob = original.pivot(index="subject_id", columns="seed", values="mean_prob_adhd").sort_index()
    condition_prob = condition.pivot(index="subject_id", columns="seed", values="mean_prob_adhd").reindex(original_prob.index)
    labels = original.drop_duplicates("subject_id").set_index("subject_id")["label"].reindex(original_prob.index).to_numpy(int)
    op, cp = original_prob.to_numpy(float), condition_prob.to_numpy(float)
    control, adhd = np.flatnonzero(labels == 0), np.flatnonzero(labels == 1)
    rng = np.random.default_rng(BOOTSTRAP_SEED + 1)
    boot = np.empty((N_BOOTSTRAP, len(METRIC_NAMES)), dtype=np.float64)
    for replicate in range(N_BOOTSTRAP):
        sampled = np.concatenate([
            rng.choice(control, size=len(control), replace=True),
            rng.choice(adhd, size=len(adhd), replace=True),
        ])
        seed_differences = []
        for seed_position in range(op.shape[1]):
            om = binary_metrics(labels[sampled], op[sampled, seed_position])
            cm = binary_metrics(labels[sampled], cp[sampled, seed_position])
            seed_differences.append([cm[m] - om[m] for m in METRIC_NAMES])
        boot[replicate] = np.nanmean(seed_differences, axis=0)

    observed = []
    for seed_position in range(op.shape[1]):
        om = binary_metrics(labels, op[:, seed_position])
        cm = binary_metrics(labels, cp[:, seed_position])
        observed.append([cm[m] - om[m] for m in METRIC_NAMES])
    observed = np.nanmean(observed, axis=0)
    lower, upper = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({
        "comparison": f"{condition_name} minus {original_name}",
        "metric": METRIC_NAMES,
        "difference": observed,
        "ci_95_lower": lower,
        "ci_95_upper": upper,
    })


def consensus_and_mcnemar(subject_predictions, original_name, condition_name):
    consensus = subject_predictions.groupby(
        ["condition", "subject_id", "label"], as_index=False
    )["mean_prob_adhd"].mean()
    consensus["prediction"] = (consensus["mean_prob_adhd"] >= PROBABILITY_THRESHOLD).astype(int)
    original = consensus[consensus["condition"] == original_name]
    condition = consensus[consensus["condition"] == condition_name]
    paired = original.merge(condition, on="subject_id", suffixes=("_original", "_condition"), validate="one_to_one")
    if not np.array_equal(paired["label_original"], paired["label_condition"]):
        raise RuntimeError("Etiquetas inconsistentes en la comparación pareada.")
    labels = paired["label_original"].to_numpy(int)
    original_correct = paired["prediction_original"].to_numpy(int) == labels
    condition_correct = paired["prediction_condition"].to_numpy(int) == labels
    original_only = int(np.sum(original_correct & ~condition_correct))
    condition_only = int(np.sum(~original_correct & condition_correct))
    discordant = original_only + condition_only
    p_value = float(binomtest(original_only, discordant, p=0.5).pvalue) if discordant else 1.0
    probability_spearman = stats.spearmanr(
        paired["mean_prob_adhd_original"], paired["mean_prob_adhd_condition"]
    ).statistic
    agreement = float(np.mean(paired["prediction_original"] == paired["prediction_condition"]))
    summary = pd.DataFrame([{
        "original_condition": original_name,
        "sensitivity_condition": condition_name,
        "n_subjects": len(paired),
        "probability_spearman": probability_spearman,
        "classification_agreement": agreement,
        "original_correct_condition_incorrect": original_only,
        "original_incorrect_condition_correct": condition_only,
        "exact_mcnemar_p": p_value,
    }])
    return consensus, summary


def top_connection_indices(vector, fraction=TOP_CONNECTION_FRACTION):
    vector = np.asarray(vector, dtype=np.float64)
    n_top = max(1, int(np.ceil(len(vector) * fraction)))
    return set(np.argpartition(vector, -n_top)[-n_top:].tolist())


def connectivity_stability(te_archive, original_name, condition_name):
    original = te_archive["original_te"]
    condition = te_archive["condition_te"]
    subject_ids = te_archive["subject_ids"].astype(str)
    labels = te_archive["labels"].astype(int)
    # La semilla no se trata como una observación clínica: primero se promedian
    # las diez representaciones de cada participante.
    original_mean = original.mean(axis=0)
    condition_mean = condition.mean(axis=0)
    rows = []
    for index, subject in enumerate(subject_ids):
        rho = stats.spearmanr(original_mean[index], condition_mean[index]).statistic
        original_top = top_connection_indices(original_mean[index])
        condition_top = top_connection_indices(condition_mean[index])
        jaccard = len(original_top & condition_top) / len(original_top | condition_top)
        rows.append({
            "subject_id": subject,
            "label": labels[index],
            "class_name": "ADHD" if labels[index] == 1 else "Control",
            "spearman_all_342_connections": float(rho),
            "jaccard_top_10_percent": float(jaccard),
        })
    by_subject = pd.DataFrame(rows)

    rng = np.random.default_rng(BOOTSTRAP_SEED + 2)
    control = np.flatnonzero(labels == 0)
    adhd = np.flatnonzero(labels == 1)
    boot = np.empty((N_BOOTSTRAP, 2), dtype=float)
    values = by_subject[["spearman_all_342_connections", "jaccard_top_10_percent"]].to_numpy(float)
    for replicate in range(N_BOOTSTRAP):
        sampled = np.concatenate([
            rng.choice(control, size=len(control), replace=True),
            rng.choice(adhd, size=len(adhd), replace=True),
        ])
        boot[replicate] = np.nanmean(values[sampled], axis=0)
    lower, upper = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    summary = pd.DataFrame([
        {
            "comparison": f"{condition_name} versus {original_name}",
            "scope": "all_participants",
            "metric": metric,
            "mean": float(np.nanmean(by_subject[column])),
            "ci_95_lower": float(lower[position]),
            "ci_95_upper": float(upper[position]),
            "n_subjects": len(by_subject),
        }
        for position, (metric, column) in enumerate([
            ("Spearman across 342 directed connections", "spearman_all_342_connections"),
            ("Jaccard for strongest 10%", "jaccard_top_10_percent"),
        ])
    ])
    class_summary = by_subject.groupby("class_name")[[
        "spearman_all_342_connections", "jaccard_top_10_percent"
    ]].agg(["mean", "std", "median", "count"])
    return by_subject, summary, class_summary


In [6]:
# ============================================================================
# 9. EJECUCIÓN COMPLETA: 10 SEMILLAS x 5 FOLDS
# ============================================================================

def run_experiment():
    set_seed(0)
    data = load_and_segment_all_subjects()
    partitions = prepare_fold_partitions(data)
    original_checkpoints = discover_original_checkpoints()

    window_frames, subject_frames, fold_metric_rows, training_rows = [], [], [], []
    te_records = {}
    original_name = "original_A1A2_same_windows"
    condition_name = EXPERIMENT_NAME

    for seed in SEEDS:
        for fold in range(N_FOLDS):
            test_indices = partitions[fold]["test"]
            condition_model, history, best_epoch, best_val_accuracy = train_condition_model(
                data, partitions, seed, fold
            )
            test_loader_condition = make_loader(
                data["X_condition"], data["y"], test_indices,
                shuffle=False, seed=seed * 100 + fold,
            )
            condition_result = evaluate_loader(condition_model, test_loader_condition, collect_te=True)
            condition_window, condition_subject, condition_te = aggregate_test_outputs(
                condition_result, data, seed, fold, condition_name
            )

            original_model = build_model().to(DEVICE)
            original_state = load_state_dict_file(original_checkpoints[(seed, fold)])
            original_model.load_state_dict(original_state, strict=True)
            test_loader_original = make_loader(
                data["X_raw"], data["y"], test_indices,
                shuffle=False, seed=seed * 100 + fold,
            )
            original_result = evaluate_loader(original_model, test_loader_original, collect_te=True)
            original_window, original_subject, original_te = aggregate_test_outputs(
                original_result, data, seed, fold, original_name
            )

            window_frames.extend([original_window, condition_window])
            subject_frames.extend([original_subject, condition_subject])
            for condition, result in ((original_name, original_result), (condition_name, condition_result)):
                metrics = binary_metrics(result["labels"], result["probabilities"])
                fold_metric_rows.append({
                    "condition": condition,
                    "seed": seed,
                    "fold": fold,
                    "n_test_windows": len(result["labels"]),
                    "n_test_subjects": len(np.unique(data["subject_ids"][test_indices])),
                    **{name: metrics[name] for name in METRIC_NAMES},
                })
            for subject, vector in original_te.items():
                te_records[(original_name, seed, subject)] = vector
            for subject, vector in condition_te.items():
                te_records[(condition_name, seed, subject)] = vector
            training_rows.append({
                "condition": condition_name,
                "seed": seed,
                "fold": fold,
                "best_epoch": best_epoch,
                "best_val_accuracy": best_val_accuracy,
                "epochs_executed": len(history),
            })

            print(
                f"COMPLETADO seed={seed} fold={fold} | "
                f"original_acc={binary_metrics(original_result['labels'], original_result['probabilities'])['accuracy']:.4f} | "
                f"{condition_name}_acc={binary_metrics(condition_result['labels'], condition_result['probabilities'])['accuracy']:.4f}"
            )
            del condition_model, original_model, condition_result, original_result
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    window_predictions = pd.concat(window_frames, ignore_index=True)
    subject_predictions = pd.concat(subject_frames, ignore_index=True)
    fold_metrics = pd.DataFrame(fold_metric_rows)
    training_summary = pd.DataFrame(training_rows)

    window_predictions.to_csv(OUTPUT_ROOT / "paired_window_predictions.csv", index=False)
    subject_predictions.to_csv(OUTPUT_ROOT / "paired_subject_predictions_by_seed.csv", index=False)
    fold_metrics.to_csv(OUTPUT_ROOT / "paired_fold_metrics.csv", index=False)
    training_summary.to_csv(OUTPUT_ROOT / "condition_training_summary.csv", index=False)

    seed_metrics = fold_metrics.groupby(["condition", "seed"], as_index=False)[METRIC_NAMES].mean()
    seed_metrics.to_csv(OUTPUT_ROOT / "window_metrics_mean_across_folds_by_seed.csv", index=False)

    participant_seed_metrics = participant_metrics_by_seed(subject_predictions)
    participant_seed_metrics.to_csv(OUTPUT_ROOT / "participant_metrics_by_seed.csv", index=False)
    participant_summary, bootstrap_distribution = stratified_participant_bootstrap(subject_predictions)
    participant_summary.to_csv(OUTPUT_ROOT / "participant_metrics_95CI.csv", index=False)
    bootstrap_distribution.to_csv(OUTPUT_ROOT / "participant_bootstrap_distribution.csv", index=False)
    metric_differences = paired_metric_difference_bootstrap(
        subject_predictions, original_name, condition_name
    )
    metric_differences.to_csv(OUTPUT_ROOT / "participant_metric_differences_95CI.csv", index=False)
    consensus, agreement = consensus_and_mcnemar(
        subject_predictions, original_name, condition_name
    )
    consensus.to_csv(OUTPUT_ROOT / "participant_consensus_predictions.csv", index=False)
    agreement.to_csv(OUTPUT_ROOT / "prediction_agreement_and_mcnemar.csv", index=False)

    ordered_subjects = np.asarray(sorted(np.unique(data["subject_ids"])), dtype=str)
    label_lookup = dict(zip(data["subject_ids"], data["y"]))
    fold_lookup = {}
    for fold, (_, _, test_subjects) in enumerate(data["folds"]):
        fold_lookup.update({subject: fold for subject in test_subjects})
    original_te_array = np.stack([
        np.stack([te_records[(original_name, seed, subject)] for subject in ordered_subjects])
        for seed in SEEDS
    ]).astype(np.float32)
    condition_te_array = np.stack([
        np.stack([te_records[(condition_name, seed, subject)] for subject in ordered_subjects])
        for seed in SEEDS
    ]).astype(np.float32)
    te_path = OUTPUT_ROOT / "paired_participant_TE_by_seed.npz"
    np.savez_compressed(
        te_path,
        original_te=original_te_array,
        condition_te=condition_te_array,
        subject_ids=ordered_subjects,
        labels=np.asarray([label_lookup[s] for s in ordered_subjects], dtype=np.int64),
        folds=np.asarray([fold_lookup[s] for s in ordered_subjects], dtype=np.int64),
        seeds=np.asarray(SEEDS, dtype=np.int64),
        original_name=np.asarray(original_name),
        condition_name=np.asarray(condition_name),
        channel_names=np.asarray(CHANNEL_NAMES),
    )
    te_archive = np.load(te_path)
    connectivity_by_subject, connectivity_summary, connectivity_by_class = connectivity_stability(
        te_archive, original_name, condition_name
    )
    connectivity_by_subject.to_csv(OUTPUT_ROOT / "connectivity_stability_by_participant.csv", index=False)
    connectivity_summary.to_csv(OUTPUT_ROOT / "connectivity_stability_95CI.csv", index=False)
    connectivity_by_class.to_csv(OUTPUT_ROOT / "connectivity_stability_by_class.csv")

    run_summary = {
        "experiment_name": EXPERIMENT_NAME,
        "preprocessing_mode": PREPROCESSING_MODE,
        "device": str(DEVICE),
        "n_seeds": len(SEEDS),
        "n_folds": N_FOLDS,
        "optuna_used": False,
        "model_params": MODEL_PARAMS,
        "training_config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "max_epochs": MAX_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        },
        "participant_metrics": participant_summary.to_dict(orient="records"),
        "paired_metric_differences": metric_differences.to_dict(orient="records"),
        "prediction_agreement": agreement.to_dict(orient="records"),
        "connectivity_stability": connectivity_summary.to_dict(orient="records"),
    }
    with open(OUTPUT_ROOT / "reviewer2_sensitivity_summary.json", "w", encoding="utf-8") as file:
        json.dump(run_summary, file, indent=2, ensure_ascii=False)

    archive_path = shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT)
    print("\n" + "=" * 88)
    print("RESULTADOS A NIVEL DE PARTICIPANTE")
    print("=" * 88)
    print(participant_summary.to_string(index=False))
    print("\nDIFERENCIAS PAREADAS (condición - original)")
    print(metric_differences.to_string(index=False))
    print("\nCONCORDANCIA DE PREDICCIONES")
    print(agreement.to_string(index=False))
    print("\nESTABILIDAD DE CONECTIVIDAD")
    print(connectivity_summary.to_string(index=False))
    print("\nResultados:", OUTPUT_ROOT)
    print("ZIP:", archive_path)
    return {
        "participant_summary": participant_summary,
        "metric_differences": metric_differences,
        "agreement": agreement,
        "connectivity_summary": connectivity_summary,
        "output_root": OUTPUT_ROOT,
        "zip": archive_path,
    }


results = run_experiment()


X raw: (8213, 19, 512)
X condición: (8213, 19, 512)
Participantes: 120
Calculando indicadores de calidad sobre las ventanas originales...
Checkpoints originales encontrados: 50
seed=0 fold=0 epoch=001 train_loss=0.5803 val_loss=0.4699 val_acc=0.7706 best=0.7706
seed=0 fold=0 epoch=005 train_loss=0.2424 val_loss=1.0814 val_acc=0.7297 best=0.7706
seed=0 fold=0 epoch=010 train_loss=0.1061 val_loss=1.4329 val_acc=0.7310 best=0.7706
COMPLETADO seed=0 fold=0 | original_acc=0.8560 | filtered_clean_A1A2_acc=0.8025
seed=0 fold=1 epoch=001 train_loss=0.4986 val_loss=0.9320 val_acc=0.6952 best=0.6952
seed=0 fold=1 epoch=002 train_loss=0.3002 val_loss=0.8259 val_acc=0.7604 best=0.7604
seed=0 fold=1 epoch=005 train_loss=0.1518 val_loss=1.1689 val_acc=0.7026 best=0.7604
seed=0 fold=1 epoch=010 train_loss=0.0817 val_loss=1.8918 val_acc=0.6974 best=0.7604
COMPLETADO seed=0 fold=1 | original_acc=0.7888 | filtered_clean_A1A2_acc=0.6155
seed=0 fold=2 epoch=001 train_loss=0.5729 val_loss=0.5146 val_acc=0.